# **실습 개요**

여러분은 한 에너지 회사의 데이터 분석팀에 합류했습니다. <br>
이 회사는 **내일의 평균 기온**을 미리 예측해서 난방 에너지 수요를 계획하려고 합니다.

앞선 실습에서는 `숫자 시퀀스 → LSTM → 마지막 hidden → 정답 하나` 구조로 숫자를 기억하는 모델을 만들었습니다. <br>
이번에는 같은 구조에 **오늘 하루의 기상 데이터**를 넣어 **내일 평균 기온(숫자 하나)** 을 예측합니다. <br>
그리고 머신러닝 수업에서 배운 **선형 회귀(Linear Regression)** 와 결과를 비교해 봅니다.

## **데이터셋 설명**

**Jena Climate Dataset**
- 독일 예나(Jena)의 기상 관측소에서 2009~2016년 동안 10분마다 측정한 기상 데이터입니다.
- 원래 14가지 측정값이 있지만, 이번 실습에서는 3가지만 사용합니다.
- 10분 간격 데이터를 **1시간 평균**으로 줄여서 사용합니다.

---
| 컬럼 | 설명 |
|---|---|
| T (degC) | 기온 (℃) |
| p (mbar) | 기압 (mbar) |
| rh (%) | 상대 습도 (%) |
---

## **예측 방식**

| | 내용 |
|---|---|
| 입력 | 오늘 하루 24시간의 기온, 기압, 습도 → `(24시간, 3개 변수)` |
| **[target]** 정답 | 내일 하루의 평균 기온 (℃) |

## **데이터셋 URL**
url : https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip

## **1. 데이터 준비하기**

데이터를 내려받고 학습, 테스트 데이터로 나눠 봅시다.
- 아래 코드를 실행하여 데이터 준비를 완료합니다.

In [ ]:
import os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


**데이터 다운로드 및 정리**

아래 코드를 실행해주세요.

In [ ]:
URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"

if not os.path.exists("jena_climate_2009_2016.csv"):
    urllib.request.urlretrieve(URL, "jena.zip")
    with zipfile.ZipFile("jena.zip") as z:
        z.extractall(".")

In [ ]:
df = pd.read_csv("jena_climate_2009_2016.csv")
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
df = df.set_index("Date Time")[["T (degC)", "p (mbar)", "rh (%)"]]
df.columns = ["temp", "pressure", "humidity"]
df.head()

,temp,pressure,humidity
Date Time,,,
2009-01-01 00:10:00,-8.02,996.52,93.3
2009-01-01 00:20:00,-8.41,996.57,93.4
2009-01-01 00:30:00,-8.51,996.53,93.9
2009-01-01 00:40:00,-8.31,996.51,94.2
2009-01-01 00:50:00,-8.27,996.51,94.1


**10분 간격 → 1시간 평균**

In [ ]:
hourly = df.resample("h").mean()
print("비어 있는 값:", hourly.isna().sum().sum())
hourly = hourly.interpolate()            # 비어 있는 시간이 있으면 앞뒤 값으로 채움
hourly.shape

비어 있는 값: 264


(70129, 3)

**하루(24시간) 단위로 묶기**

- 1시간 단위 데이터를 24개씩 묶으면 "하루"가 됩니다.
- 이렇게 하면 모델에 넣을 데이터가 `(하루 수, 24시간, 3개 변수)` 모양이 됩니다.

In [ ]:
n_days = len(hourly) // 24                     # 완전한 하루의 개수
hourly = hourly.iloc[: n_days * 24]            # 남는 시간은 버림
X_all = hourly.values.reshape(n_days, 24, 3).astype(np.float32)
X_all.shape

(2922, 24, 3)

**입력과 정답 만들기**

- 입력: 오늘 하루 (24시간, 3개 변수)
- 정답: **내일** 하루의 평균 기온

In [ ]:
daily_mean = X_all[:, :, 0].mean(axis=1)       # 하루 평균 기온
X = X_all[:-1]                                 # 오늘 (마지막 날은 내일이 없어서 제외)
y = daily_mean[1:]                             # 내일 평균 기온
print(X.shape, y.shape)

(2921, 24, 3) (2921,)


**학습, 테스트 데이터 분리**

- 시간 순서대로 **앞 80%는 학습, 뒤 20%는 테스트**로 나눕니다.
- 무작위로 섞으면 미래 데이터로 학습해서 과거를 맞히는 셈이 되기 때문입니다.

In [ ]:
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

## **2. 데이터 살펴보기**

**Q1. df를 출력해 데이터를 DataFrame으로 변환한 형태를 확인합니다.**

In [ ]:
# code here

**Q2. Data의 shape를 확인하세요.**

X_train, X_test, y_train, y_test의 shape를 출력해보고, 각 숫자가 무엇을 의미하는지 생각해 봅시다.

In [ ]:
# code here

## **2. 데이터 시각화하기**

**Q1. 하루 평균 기온을 시간 순서대로 선그래프로 그려 보세요.**

```
df.index.date
```
df의 index(날짜)를 datetime.date 객체로 가져올 수 있습니다.

In [ ]:
# code here

**Q2. 시간(0~23시)별 평균 기온을 선그래프로 그려 보세요.**
- X_train에서 [0]번째의 정보는 temp입니다. (기온)
- 여기서 평균을 "가로방향"으로 구해주면 됩니다.

In [ ]:
# code here

**Q3. 오늘 평균 기온(x)과 내일 평균 기온(y)의 관계를 산점도로 그려주세요.**

- 오늘 평균 기온은 `X_train`의 기온(0번 변수)를 '새로방향'으로 평균내면 됩니다.

In [ ]:
# code here

## **3. 데이터 전처리하기**

### **3-1. 정규화**

**Q1. 입력(X)과 정답(y)을 평균 0, 표준편차 1이 되도록 정규화하세요.**

- 평균과 표준편차는 **train 데이터로만** 계산하고, test에도 같은 값을 적용하세요
- (-1, 3)으로 형태 변형을 했다가 다시 data.shape로 변환해주면 (T,B,F) 형태가 유지됩니다.
- LSTM을 학습하기 위해서는 연속값을 예측할때에는 정답 (y)도 scaling을 해줘야 합니다. (LSTM은 큰 값을 내기 어렵습니다.)

In [ ]:
# code here

### **3-2. DataLoader 만들기 (LSTM용)**

**Q1. 정규화한 데이터를 텐서로 바꾸고, 배치 크기 32의 DataLoader를 만드세요.**
- train data는 shuffle하되, test data는 shuffle 하지 않습니다.

In [ ]:
# code here

**Q2. 배치 하나를 꺼내서 shape을 확인하세요.**

In [ ]:
# code here

## **4. 선형 회귀 모델**

**Q1. 입력을 (샘플수, 24x3) 모양으로 펴 주세요**
- 선형 회귀 모델은 3차원 데이터를 입력받을 수 없습니다.
```
reshape(len(data_scaled), -1)
```

In [ ]:
# code here

**Q2. 선형 회귀 모델을 만들어주세요.**

In [ ]:
# code here

**Q3. 선형 회귀 모델을 학습시켜주세요.**

In [ ]:
# code here

**Q4. test 데이터로 내일 평균 기온을 예측하고, 오차를 단위로 구하세요.**

- 예측값은 정규화된 값이니 `y_scaler.inverse_transform`으로 ℃로 되돌린 뒤 정답(`y_test`)과 비교합니다.
- RMSE를 활용합니다.

In [ ]:
# code here

## **5. LSTM 모델**

**Q1. LSTM 모델을 만들어 주세요.**
- 입력 -> LSTM -> 출력
- 앞 실습과 같은 구조입니다. 입력 차원은 3, 출력은 **숫자 1개**(내일 평균 기온)입니다.

In [ ]:
# code here
class LSTM(nn.Module):
  def __init__(self):
    pass

  def forward(self, x):
    pass

**Q2. 모델을 생성하고, 출력하여 층 구조를 확인합니다. 그리고 배치 하나를 넣어 출력 shape을 확인하세요.**
- 신경망 모델은 GPU로 옮겨주세요.

In [ ]:
# code here

LSTM(
  (lstm): LSTM(3, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)
torch.Size([32, 1])


**Q3. 예측 함수 predict와 오차(RMSE)를 계산하는 evaluate 함수를 만드세요.**
- 배치 데이터는 학습시 GPU로 옮겨주세요.

In [ ]:
# code here

**Q4. 학습을 진행하는 train 함수를 만드세요.**
- 모델, epochs, learning rate를 파라미터로 받아줍니다.
- 손실 함수 : RMSELoss
- 최적화 : Adam
- learning rate : 0.001
- epochs : 100

epoch별 loss와 RMSE 값들을 리스트에 저장하여 return합니다.

In [ ]:
# code

**Q5. LSTM을 학습시킵니다.**

In [ ]:
# code here

## **6. 결과 비교**

**Q1. LSTM의 train, test loss를 epoch별로 그래프로 그리세요.**

In [ ]:
# code here

**Q2. 기준선, 선형 회귀, LSTM의 test MAE를 막대그래프로 비교하세요.**

- **기준선**은 아무 모델 없이 "오늘 평균 기온 = 내일 평균 기온"이라고 예측하는 가장 단순한 방법입니다.
- 모델이 뭔가 배웠다고 말하려면 기준선보다 오차가 작아야 합니다.

그리고 이 그래프로 알 수 있는 정보를 코드 아래 적어 주세요.

In [ ]:
# code here

**Q3. test 데이터의 마지막 60일에 대해 실제 기온과 두 모델의 예측을 한 그래프에 그리세요.**

그리고 이 그래프로 알 수 있는 정보를 코드 아래 적어 주세요.

In [ ]:
# code here